In [1]:
import random
import pandas as pd

# ==== ĐẶC TRƯNG NỀN ====
residence_area = {
    'high': ["TP Hồ Chí Minh", "Hà Nội", "Hải Phòng", "Quảng Ninh", "Đà Nẵng",
             "Lạng Sơn", "Lào Cai", "Tây Ninh", "Kiên Giang", "Bình Dương",
             "Đồng Nai", "Cần Thơ", "An Giang", "Bà Rịa - Vũng Tàu"],
    'medium': ["Bình Thuận", "Bình Phước", "Bình Định", "Khánh Hòa", "Nghệ An",
               "Thanh Hóa", "Thừa Thiên Huế", "Đắk Lắk", "Đắk Nông", "Gia Lai",
               "Kon Tum", "Quảng Nam", "Quảng Ngãi", "Phú Yên", "Hà Tĩnh",
               "Hải Dương", "Nam Định", "Ninh Bình", "Thái Nguyên", "Vĩnh Phúc",
               "Long An", "Hậu Giang", "Tiền Giang", "Trà Vinh", "Vĩnh Long",
               "Sóc Trăng", "Cà Mau", "Bạc Liêu", "Bến Tre", "Phú Thọ",
               "Hưng Yên", "Thái Bình", "Ninh Thuận", "Hòa Bình", "Yên Bái",
               "Tuyên Quang", "Hà Nam", "Lâm Đồng"],
    'low': ["Bắc Giang", "Bắc Kạn", "Bắc Ninh", "Cao Bằng", "Điện Biên",
            "Hà Giang", "Sơn La", "Lai Châu", "Quảng Trị", "Đồng Tháp"]
}

occupation = {
    'high': ["Cầm đồ", "Kinh doanh nhà hàng karaoke", "Chủ quán bar", "Làm từ thiện",
             "Tiếp viên quán", "Vũ công tự do", "Streamer", "Youtuber", "Tiktoker",
             "Kinh doanh vàng bạc", "Chơi chứng khoán", "Doanh nhân", "Tự doanh",
             "Môi giới bất động sản", "Kinh doanh đa cấp", "Không rõ"],
    'medium': ["Luật sư", "Nhân viên ngân hàng", "Kế toán", "Kiểm toán viên",
               "Chuyên viên tài chính", "Tư vấn bảo hiểm", "Môi giới chứng khoán",
               "Tài xế giao dịch tiền", "Nhà báo", "Nghệ sĩ tự do", "Ca sĩ", "Diễn viên",
               "MC", "Freelancer", "Nhà văn", "Nhiếp ảnh gia", "Thiết kế đồ họa",
               "Chuyên gia SEO", "Quản trị fanpage", "Lái xe công nghệ", "Bán hàng online",
               "Chủ cửa hàng", "Lái taxi", "Chủ quán ăn", "Nhân viên marketing",
               "Cửa hàng trưởng", "Quản lý khách sạn", "Làm thuê thời vụ", "Trình dược viên",
               "Sinh viên", "Thất nghiệp", "Nội trợ"],
    'low': ["Công an", "Bộ đội", "Thẩm phán", "Kiểm sát viên", "Giảng viên đại học",
            "Giáo viên phổ thông", "Gia sư", "Nhà khoa học", "Kỹ sư phần mềm", "Lập trình viên",
            "Kỹ sư xây dựng", "Kỹ sư điện", "Kỹ thuật viên phòng lab", "Chuyên viên CNTT",
            "Phân tích dữ liệu", "Bác sĩ", "Y tá", "Dược sĩ", "Bác sĩ thú y", 
            "Nhân viên chăm sóc sắc đẹp", "Chuyên viên spa", "Chăm sóc người già", "Công nhân",
            "Thợ xây", "Thợ điện", "Thợ nước", "Thợ mộc", "Thợ hàn", "Lái xe tải", "Bốc vác",
            "Bảo vệ", "Nhân viên bảo trì", "Nhân viên bán hàng", "Nhân viên phục vụ", "Lễ tân",
            "Nhân viên thu ngân", "Tư vấn tuyển sinh", "Học sinh"]
}

def get_label(score):
    if score >= 17:
        return 4
    elif score >= 13:
        return 3
    elif score >= 9:
        return 2
    elif score >= 5:
        return 1
    else:
        return 0

def calculate_score(per_violation_score, per_role_score, per_legal, org_violation, org_legal, alias_count, age, area, job):
    score = 0
    reasons = []

    # --- VI PHẠM CÁ NHÂN ---
    if per_legal == "Minh oan":
        reasons.append("Được minh oan (reset vi phạm và vai trò)")
    else:
        if per_violation_score > 0:
            score += per_violation_score
            reasons.append(f"Vi phạm: +{per_violation_score}")
        if per_role_score > 0:
            score += per_role_score
            reasons.append(f"Vai trò: +{per_role_score}")
        if per_legal == "Đã kết án":
            score += 3; reasons.append("Đã kết án (+3)")
        elif per_legal in ["Đang điều tra", "Truy tố"]:
            score += 2; reasons.append("Đang điều tra/truy tố (+2)")
        elif per_legal == "Chưa rõ":
            score += 1; reasons.append("Pháp lý chưa rõ (+1)")

    # --- VI PHẠM TỔ CHỨC ---
    if org_violation:
        if "tài chính" in org_violation or "cấm vận" in org_violation:
            score += 5; reasons.append("Tổ chức bị chế tài mạnh (+5)")
        elif "ngành" in org_violation:
            score += 4; reasons.append("Tổ chức bị hạn chế ngành (+4)")
        elif "thứ cấp" in org_violation:
            score += 3; reasons.append("Tổ chức bị trừng phạt thứ cấp (+3)")

    if org_legal == "Đã kết án":
        score += 3; reasons.append("Tổ chức đã kết án (+3)")
    elif org_legal in ["Đang điều tra", "Truy tố"]:
        score += 2; reasons.append("Tổ chức bị điều tra/truy tố (+2)")

    # --- ALIAS + TUỔI ---
    if alias_count > 3:
        score += 2; reasons.append("Nhiều alias (+2)")
    elif alias_count > 0:
        score += 1; reasons.append("Có alias (+1)")
    if age < 30 or age > 65:
        score += 1; reasons.append("Tuổi rủi ro (+1)")

    # --- ĐIỂM NỀN ---
    if area in residence_area['high']:
        score += 2; reasons.append("Khu vực rủi ro cao (+2)")
    elif area in residence_area['medium']:
        score += 1; reasons.append("Khu vực rủi ro vừa (+1)")

    if job in occupation['high']:
        score += 3; reasons.append("Nghề rủi ro cao (+3)")
    elif job in occupation['medium']:
        score += 1; reasons.append("Nghề rủi ro vừa (+1)")

    return score, reasons

# ==== HÀM CHÍNH ====
def generate_aml_data_realistic_v2_4(n_samples=1000, seed=42):
    random.seed(seed)
    samples = []

    # Phân phối mục tiêu
    target_distribution = {
        4: int(n_samples * random.uniform(0.04, 0.06)),
        3: int(n_samples * random.uniform(0.08, 0.12)),
        2: int(n_samples * random.uniform(0.15, 0.25)),
        1: int(n_samples * random.uniform(0.25, 0.35)),
    }
    target_distribution[0] = n_samples - sum(target_distribution.values())

    area_list = [loc for sub in residence_area.values() for loc in sub]
    job_list = [job for sub in occupation.values() for job in sub]

    violation_type_pool = {
        6: ["Rửa tiền", "Tài trợ khủng bố"],
        4: ["Lừa đảo", "Chiếm đoạt tài sản", "Tham nhũng", "Hối lộ"],
        2: ["Vi phạm hành chính", "Tranh chấp dân sự", "Vi phạm dân sự", "Vi phạm nhỏ"],
        0: [None]
    }
    role_pool = {
        6: ["Chủ mưu", "Cầm đầu", "Tổ chức thực hiện"],
        4: ["Tham gia", "Giúp sức", "Đồng phạm"],
        2: ["Bị nhắc tên", "Liên quan bị động"],
        0: [None]
    }
    
    non_null_sample_count = 0
    for _ in range(n_samples * 3):
        area = random.choice(area_list)
        job = random.choice(job_list)
        alias_count = random.choices([0, 1, 2, 3, 4], weights=[0.5, 0.2, 0.15, 0.1, 0.05])[0]
        age = random.randint(18, 75)

        # per_violation_score = random.choices([6, 4, 2, 0], weights=[0.04, 0.08, 0.15, 0.73])[0]
        per_violation_score = random.choices([6, 4, 2, 0], weights=[0.1, 0.15, 0.3, 0.45])[0]
        per_violation = random.choice(violation_type_pool[per_violation_score])
        if per_violation_score > 0:
            per_role_score = random.choices([6, 4, 2], weights=[0.3, 0.4, 0.3])[0]
            per_role = random.choice(role_pool[per_role_score])
            per_legal = random.choices(
                ["Đã kết án", "Đang điều tra", "Truy tố", "Chưa rõ", "Minh oan", None],
                weights=[0.03, 0.05, 0.04, 0.1, 0.01, 0.77])[0]
        else:
            per_role_score = 0
            per_role = None
            per_legal = None

        if random.random() < 0.3: #0.15
            org_violation = random.choice([
                "Trừng phạt tài chính", "Cấm vận kinh tế",
                "Trừng phạt ngành", "Trừng phạt thứ cấp"
            ])
            org_legal = random.choice(["Đã kết án", "Đang điều tra", "Truy tố", None])
        else:
            org_violation, org_legal = None, None

        score, reasons = calculate_score(
            per_violation_score, per_role_score, per_legal,
            org_violation, org_legal,
            alias_count, age, area, job
        )

        label = get_label(score)
        if sum(1 for s in samples if s["label"] == label) >= target_distribution[label]:
            continue

        is_non_null = any([
            per_violation_score > 0,
            per_role_score > 0,
            per_legal != "",
            org_violation != "",
            org_legal != ""
        ])
        
        # Nếu tất cả đều rỗng mà số mẫu non-null chưa đủ, thì bỏ qua mẫu này
        if not is_non_null and non_null_sample_count < int(n_samples * 0.7):
            continue  # ép tạo lại mẫu khác
        elif is_non_null:
            non_null_sample_count += 1
    
        samples.append({
            "residence_area": area,
            "occupation": job,
            "alias_count": alias_count,
            "age": age,
            "per_violation_type": per_violation,
            "per_role": per_role,
            "per_legal_status": per_legal,
            "org_violation_type": org_violation,
            "org_legal_status": org_legal,
            "total_score": score,
            "label": label,
            "risk_reason": "; ".join(reasons)
        })

        if len(samples) >= n_samples:
            break

    return pd.DataFrame(samples)


In [2]:
df = generate_aml_data_realistic_v2_4(n_samples=1000, seed=42)
df.to_csv("aml_dataset_v2_3.csv", index=False)

In [3]:
df['label'].value_counts()

label
0    418
1    272
2    177
3     81
4     52
Name: count, dtype: int64

In [4]:
df['org_legal_status'].isna().sum()

808

In [5]:
df.head()

,residence_area,occupation,alias_count,age,per_violation_type,per_role,per_legal_status,org_violation_type,org_legal_status,total_score,label,risk_reason
0,Hòa Bình,Môi giới bất động sản,1,75,Vi phạm nhỏ,Chủ mưu,Chưa rõ,None,None,15,3,Vi phạm: +2; Vai trò: +6; Pháp lý chưa rõ (+1)...
1,Hà Nội,Thợ xây,0,59,None,None,None,Trừng phạt ngành,Đã kết án,9,2,Tổ chức bị hạn chế ngành (+4); Tổ chức đã kết ...
2,Yên Bái,Chuyên viên tài chính,1,39,Tranh chấp dân sự,Liên quan bị động,Truy tố,None,None,9,2,Vi phạm: +2; Vai trò: +2; Đang điều tra/truy t...
3,Đắk Nông,Trình dược viên,1,69,Tài trợ khủng bố,Giúp sức,Đang điều tra,Trừng phạt ngành,Đang điều tra,22,4,Vi phạm: +6; Vai trò: +4; Đang điều tra/truy t...
4,Thái Bình,Tiktoker,0,32,None,None,None,None,None,4,0,Khu vực rủi ro vừa (+1); Nghề rủi ro cao (+3)
